In [ ]:
# OBELiX PRODUCTION-STRUCTURE / SPECTRUM AUDIT — SINGLE GOOGLE COLAB CELL
# Purpose
# Audit every production Option-A MatterSim–Phonopy spectrum against the
# intended OBELiX composition and the actual ordered pre-relaxation structure
# used by the production workflow. The cell is read-only: it does not modify
# any production structure or spectrum.
#
# Main outputs
# ------------
# 1. production_structure_composition_audit.csv
# 2. all_structure_candidates.csv
# 3. source_to_production_ordering_audit.csv
# 4. spectrum_grid_qc.csv
# 5. eigenfrequency_stability_qc.csv
# 6. regeneration_priority.csv
# 7. missing_or_ambiguous_files.csv
# 8. summary tables, publication-quality PDF/PNG figures, and a ZIP archive
#
# Notes
# -----
# * "True imaginary mode" diagnostics come only from unbroadened mesh
#   frequencies in phonon_eigen_data.npz. A negative plotted-DOS grid endpoint
#   is never interpreted as a phonon instability.
# * Composition matching is scale invariant: a cell containing four formula
#   units of Li3PO4 is accepted as Li3PO4.
# * Fractional OBELiX compositions are compared with a finite-cell-aware hybrid
#   criterion. A one-atom rounding of a dilute dopant is not treated as a 100%
#   chemical failure merely because the denominator is small.
# * Production-to-source integrity is checked against ordered_adaptive_<ID>.cif,
#   which is the structure actually passed into MatterSim relaxation. Generic
#   relaxed CIFs found elsewhere in Drive are listed as candidates but are not
#   automatically treated as the production source.
# * A dedicated framework-scaled Li error still identifies vacancy overfilling,
#   e.g. Li16P4S16 versus expected Li12P4S16 for four Li3PS4 units.
# ------------------------------- CONFIGURATION -------------------------------
from pathlib import Path

OUTPUT_DRIVE_DIR = Path("/content/drive/MyDrive/OBELiX_composition_audit")
LOCAL_WORKDIR = Path("/content/obelix_composition_audit")

# Shared-folder IDs used by the prior DOS/PDOS notebooks.
PARENT_FOLDER_ID = "1kT0ZT9i2JiVBNd3mbOY3m9iOOvpSa9dm"
TRAIN_PDOS_ID = "1MB6BoBrl1fe9r7k-uSWkTd1W5XsDrSkH"
TEST_PDOS_ID = "1s_34z1FvR87R8iCccfMZTIZPXE_i9aGr"

TRAIN_METADATA_URL = (
    "https://raw.githubusercontent.com/NRC-Mila/OBELiX/main/data/downloads/train.csv"
)
TEST_METADATA_URL = (
    "https://raw.githubusercontent.com/NRC-Mila/OBELiX/main/data/downloads/test.csv"
)

# Optional additional roots holding input CIFs. Existing paths are used;
# nonexistent paths are ignored. Add project-specific paths here if necessary.
EXTRA_SEARCH_ROOTS = [
    "/content/drive/MyDrive/Train_cif",
    "/content/drive/MyDrive/Test_cif",
    "/content/drive/MyDrive/train_cif",
    "/content/drive/MyDrive/test_cif",
    "/content/drive/MyDrive/phonon_results",
]

# Manual overrides are optional. Use lowercase OBELiX IDs.
# Example:
# STRUCTURE_OVERRIDES = {
#     "9lo": {
#         "production": "/content/drive/MyDrive/.../ultra_relaxed_generated.cif",
#         "source": "/content/drive/MyDrive/.../ordered_adaptive_9lo.cif",
#         "eigen": "/content/drive/MyDrive/.../phonon_eigen_data.npz",
#         "spectrum": "/content/drive/MyDrive/.../phonon_dos_pdos.csv",
#     }
# }
STRUCTURE_OVERRIDES = {}

# Composition tolerances.
# Relative errors alone are inappropriate for dilute dopants: changing an
# expected count from 0.8 to the only feasible integer count 1 is a 25% relative
# error but only a 0.2-atom finite-cell rounding. We therefore require either a
# small absolute count residual OR a small relative residual for each element.
PASS_MAX_RELATIVE_STOICH_ERROR = 0.05       # 2%
WARN_MAX_RELATIVE_STOICH_ERROR = 0.1       # 5%
PASS_MAX_ABSOLUTE_COUNT_ERROR_ATOMS = 0.51 # nearest-integer realization
WARN_MAX_ABSOLUTE_COUNT_ERROR_ATOMS = 1.01 # one-atom ordering approximation
PASS_TOTAL_VARIATION_ERROR = 0.05          # 2% composition-fraction distance
WARN_TOTAL_VARIATION_ERROR = 0.1          # 6% composition-fraction distance
LI_PASS_MAX_ABSOLUTE_COUNT_ERROR_ATOMS = 0.51
LI_WARN_MAX_ABSOLUTE_COUNT_ERROR_ATOMS = 1.01
LI_STOICHIOMETRY_FAIL_THRESHOLD = 0.05     # retained for plotting/reporting
ANCHOR_MIN_EXPECTED_COUNT = 1.0
ANCHOR_MIN_FRACTION_OF_MAX = 0.05

# Geometry and spectral QC thresholds.
SEVERE_MIN_DISTANCE_A = 0.70
WARNING_MIN_DISTANCE_A = 1.00
COARSE_DOS_GRID_THz = 1.00
HIGH_MAX_FREQUENCY_THz = 150.0
VERY_HIGH_MAX_FREQUENCY_THz = 300.0
IMAGINARY_THRESHOLDS_THz = (-0.10, -0.50, -1.00)

# Optional oxidation-state feasibility heuristic. This asks only whether
# pymatgen can find at least one charge-neutral oxidation-state assignment;
# it is not proof of the true oxidation states.
RUN_OXIDATION_STATE_HEURISTIC = True
MAX_ELEMENTS_FOR_OXIDATION_GUESS = 7

# Parsing and output settings.
MAX_STRUCTURE_CANDIDATES_PER_ID = 30
MIN_AUTOMATIC_PRODUCTION_SCORE = 300
MIN_AUTOMATIC_SOURCE_SCORE = 400
FIGURE_DPI = 600
ZIP_NAME = "OBELiX_composition_audit_outputs.zip"

# ----------------------------- PACKAGE INSTALLATION --------------------------
import sys, subprocess
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "pymatgen>=2025.6.14", "pyyaml>=6.0", "matplotlib>=3.8", "pandas>=2.0"
])

# ---------------------------------- IMPORTS ----------------------------------
import os
import re
import json
import math
import lzma
import gzip
import shutil
import zipfile
import warnings
import traceback
from collections import defaultdict, Counter
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt

from pymatgen.core import Composition, Structure, Lattice

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ------------------------------- OUTPUT FOLDERS ------------------------------
if LOCAL_WORKDIR.exists():
    shutil.rmtree(LOCAL_WORKDIR)
LOCAL_WORKDIR.mkdir(parents=True, exist_ok=True)

TABLES = LOCAL_WORKDIR / "tables"
FIGURES = LOCAL_WORKDIR / "figures"
LOGS = LOCAL_WORKDIR / "logs"
for folder in (TABLES, FIGURES, LOGS):
    folder.mkdir(parents=True, exist_ok=True)

# ------------------------------ MOUNT GOOGLE DRIVE ---------------------------
from google.colab import drive

drive.mount("/content/drive", force_remount=True)
MYDRIVE_ROOT = Path("/content/drive/MyDrive")
SHORTCUT_ROOT = Path("/content/drive/.shortcut-targets-by-id")

# ----------------------------------------------------------------------
# Your current Google Drive folders.
# Each material subfolder, e.g. 122/, aab/, 6ji/, contains BOTH:
#   Lithium_PDOS_<material_id>.csv
#   Total_DOS_<material_id>.csv
# ----------------------------------------------------------------------
TRAIN_PDOS_ROOT = Path("/content/drive/MyDrive/MLIP-main/DOSFILES/Train_cif")
TEST_PDOS_ROOT  = Path("/content/drive/MyDrive/MLIP-main/DOSFILES/Test_cif")

# Temporary combined Option-A CSVs are written here. Production Drive folders
# are only read, not modified.
COMBINED_SPECTRA_DIR = LOCAL_WORKDIR / "combined_option_a_spectra"
COMBINED_SPECTRA_DIR.mkdir(parents=True, exist_ok=True)
SPECTRUM_SOURCE_DIRS = {}


def usable_directory(path):
    path = Path(path)
    try:
        return path.exists() and path.is_dir() and os.access(path, os.R_OK)
    except Exception:
        return False


def show_directory(path, max_entries=100):
    path = Path(path)
    print(f"\nContents of: {path}")
    if not usable_directory(path):
        print("  [not accessible]")
        return
    try:
        entries = sorted(path.iterdir(), key=lambda p: p.name.lower())
        for p in entries[:max_entries]:
            kind = "DIR " if p.is_dir() else "FILE"
            print(f"  {kind}  {repr(p.name)}")
        if len(entries) > max_entries:
            print(f"  ... and {len(entries) - max_entries} more entries")
    except Exception as exc:
        print(f"  Could not list directory: {exc}")


def check_spectrum_root(root, label):
    root = Path(root)
    if not usable_directory(root):
        print("\nTop-level MyDrive folders:")
        show_directory(MYDRIVE_ROOT, max_entries=100)
        print("\nVisible shortcut-target IDs:")
        show_directory(SHORTCUT_ROOT, max_entries=100)
        raise FileNotFoundError(
            f"{label} folder not found or not readable:\n{root}\n\n"
            "If this is a shared folder, add a shortcut to My Drive or run Colab "
            "with the Google account that has access."
        )
    li_files = sorted(root.rglob("Lithium_PDOS_*.csv"))
    total_files = sorted(root.rglob("Total_DOS_*.csv"))
    print(f"{label}: {root}")
    print(f"  Li PDOS files: {len(li_files)}")
    print(f"  Total DOS files: {len(total_files)}")
    if not li_files:
        raise RuntimeError(f"No Lithium_PDOS_*.csv files found in {root}")
    if not total_files:
        raise RuntimeError(f"No Total_DOS_*.csv files found in {root}")


check_spectrum_root(TRAIN_PDOS_ROOT, "TRAIN")
check_spectrum_root(TEST_PDOS_ROOT, "TEST")

# ------------------------------- LOAD METADATA -------------------------------
def load_metadata(url, split):
    df = pd.read_csv(url, dtype={"ID": str})
    df["ID"] = df["ID"].astype(str).str.strip().str.lower()
    df["split"] = split
    return df


train_meta = load_metadata(TRAIN_METADATA_URL, "train")
test_meta = load_metadata(TEST_METADATA_URL, "test")
metadata = pd.concat([train_meta, test_meta], ignore_index=True)
metadata = metadata.drop_duplicates(subset=["split", "ID"], keep="first")
meta_by_id = metadata.set_index(["split", "ID"], drop=False)
known_ids = set(metadata["ID"].astype(str))

# ------------------------------ GENERIC UTILITIES ----------------------------
def normalize_id(value):
    return str(value).strip().lower()


def extract_obelix_id(path, collection_root, known):
    path = Path(path)
    try:
        rel_parts = path.relative_to(collection_root).parts
    except Exception:
        rel_parts = path.parts

    for part in reversed(rel_parts):
        stem = Path(part).stem.lower()
        tokens = re.findall(r"(?<![a-z0-9])([a-z0-9]{3})(?![a-z0-9])", stem)
        for token in tokens:
            if token in known:
                return token
        if stem in known:
            return stem

    for part in reversed(rel_parts):
        stem = Path(part).stem.lower()
        for token in re.split(r"[^a-z0-9]+", stem):
            if token in known:
                return token
        for oid in known:
            if (
                stem.startswith(oid + "_") or stem.startswith(oid + "-")
                or stem.endswith("_" + oid) or stem.endswith("-" + oid)
            ):
                return oid
    return None


def path_distance(a, b):
    a = Path(a).resolve()
    b = Path(b).resolve()
    a_parts = a.parts
    b_parts = b.parts
    common = 0
    for x, y in zip(a_parts, b_parts):
        if x != y:
            break
        common += 1
    return (len(a_parts) - common) + (len(b_parts) - common)


def json_counts(counts):
    return json.dumps(
        {str(k): round(float(v), 8) for k, v in sorted(counts.items())},
        sort_keys=True,
    )


def finite_or_nan(value):
    try:
        x = float(value)
        return x if np.isfinite(x) else np.nan
    except Exception:
        return np.nan

# -------------------------- FIND OPTION-A SPECTRA ----------------------------
def normalized_columns(columns):
    return {re.sub(r"[^a-z0-9]+", "", str(c).lower()): c for c in columns}


def _pick_column(df, possible_keys, label, path):
    cmap = normalized_columns(df.columns)
    for key in possible_keys:
        if key in cmap:
            return cmap[key]
    raise KeyError(f"Could not find {label} column in {path}. Columns: {list(df.columns)}")


def _read_total_csv(path):
    df = pd.read_csv(path)
    fcol = _pick_column(df, ["frequencythz", "frequency", "freqthz", "freq"], "frequency", path)
    ycol = _pick_column(df, ["totaldos", "dos", "densityofstates"], "Total_DOS", path)
    out = pd.DataFrame({
        "Frequency_THz": pd.to_numeric(df[fcol], errors="coerce"),
        "Total_DOS": pd.to_numeric(df[ycol], errors="coerce"),
    }).replace([np.inf, -np.inf], np.nan).dropna()
    return out.sort_values("Frequency_THz").groupby("Frequency_THz", as_index=False).mean(numeric_only=True)


def _read_li_csv(path):
    df = pd.read_csv(path)
    fcol = _pick_column(df, ["frequencythz", "frequency", "freqthz", "freq"], "frequency", path)
    ycol = _pick_column(df, ["lithium_pdos", "lipdos", "liprojecteddos", "lithiumpdos", "lithiumdos", "lidos"], "Li_PDOS", path)
    out = pd.DataFrame({
        "Frequency_THz": pd.to_numeric(df[fcol], errors="coerce"),
        "Li_PDOS": pd.to_numeric(df[ycol], errors="coerce"),
    }).replace([np.inf, -np.inf], np.nan).dropna()
    return out.sort_values("Frequency_THz").groupby("Frequency_THz", as_index=False).mean(numeric_only=True)


def combine_total_and_li_csv(total_path, li_path, out_path):
    total = _read_total_csv(total_path)
    li = _read_li_csv(li_path)
    freq = total["Frequency_THz"].to_numpy(float)
    total_y = total["Total_DOS"].to_numpy(float)
    li_freq = li["Frequency_THz"].to_numpy(float)
    li_y = li["Li_PDOS"].to_numpy(float)

    if len(freq) == len(li_freq) and np.allclose(freq, li_freq, rtol=1e-8, atol=1e-10):
        li_on_total = li_y
    else:
        li_on_total = np.interp(freq, li_freq, li_y, left=0.0, right=0.0)

    combined = pd.DataFrame({
        "Frequency_THz": freq,
        "Total_DOS": np.clip(total_y, 0, None),
        "Li_PDOS": np.clip(li_on_total, 0, None),
    })
    out_path.parent.mkdir(parents=True, exist_ok=True)
    combined.to_csv(out_path, index=False)
    return out_path


def inspect_option_a_csv(path):
    try:
        header = pd.read_csv(path, nrows=3)
    except Exception:
        return None
    cmap = normalized_columns(header.columns)
    aliases = {
        "frequency": ["frequencythz", "frequency", "freqthz"],
        "total": ["totaldos", "totaldensityofstates"],
        "li": ["lipdos", "liprojecteddos", "lidos"],
    }
    chosen = {}
    for key, names in aliases.items():
        match = next((cmap[n] for n in names if n in cmap), None)
        if match is None:
            return None
        chosen[key] = match
    return chosen


def option_a_score(path):
    text = str(path).lower()
    name = path.name.lower()
    score = 500
    if "phonon" in name:
        score += 50
    if "pdos" in name or "projected" in name:
        score += 50
    if "dos" in name:
        score += 20
    if "raw_spectra" in text or "convergence" in text or "reference" in text:
        score -= 300
    if "backup" in text or "old" in text:
        score -= 50
    return score


def _canonical_spectrum_file(files, oid, prefix):
    exact = f"{prefix}_{oid}.csv".lower()
    for path in files:
        if path.name.lower() == exact:
            return path
    token_matches = [p for p in files if re.search(rf"(?:^|[_-]){re.escape(oid)}(?:[_-]|\.csv$)", p.name.lower())]
    return token_matches[0] if token_matches else files[0]


def spectrum_folder_score(folder, oid, li_path, total_path):
    text = str(folder).lower()
    score = 0.0
    if folder.name.lower() == oid:
        score += 1000
    if li_path.name.lower() == f"lithium_pdos_{oid}.csv":
        score += 300
    if total_path.name.lower() == f"total_dos_{oid}.csv":
        score += 300
    if (folder / f"relaxed_{oid}.cif").exists():
        score += 250
    if (folder / f"ordered_adaptive_{oid}.cif").exists():
        score += 200
    if (folder / "phonon_eigen_data.npz").exists():
        score += 100
    if any(term in text for term in ["raw_spectra", "convergence", "reference", "nims"]):
        score -= 1500
    if any(term in text for term in ["backup", "archive", "old", "copy"]):
        score -= 500
    # Prefer shallower canonical material folders when everything else ties.
    score -= 0.01 * len(folder.parts)
    return score


def index_option_a_spectra(root, split):
    selected = {}
    rows = []
    root = Path(root)
    grouped = defaultdict(list)

    candidate_dirs = sorted({p.parent for p in root.rglob("*.csv")})
    for folder in candidate_dirs:
        oid = extract_obelix_id(folder, root, known_ids)
        if oid is None:
            for p in sorted(folder.glob("*.csv")):
                oid = extract_obelix_id(p, root, known_ids)
                if oid is not None:
                    break
        if oid is None:
            continue

        li_files = sorted(folder.glob("Lithium_PDOS_*.csv"))
        total_files = sorted(folder.glob("Total_DOS_*.csv"))
        if not li_files or not total_files:
            continue

        li_path = _canonical_spectrum_file(li_files, oid, "Lithium_PDOS")
        total_path = _canonical_spectrum_file(total_files, oid, "Total_DOS")
        grouped[oid].append({
            "folder": folder,
            "li_path": li_path,
            "total_path": total_path,
            "score": spectrum_folder_score(folder, oid, li_path, total_path),
        })

    for oid, candidates in sorted(grouped.items()):
        ranked = sorted(candidates, key=lambda x: (x["score"], str(x["folder"])), reverse=True)
        top_score = ranked[0]["score"]
        second_score = ranked[1]["score"] if len(ranked) > 1 else np.nan
        for rank, item in enumerate(ranked, start=1):
            is_selected = rank == 1
            combined_path = COMBINED_SPECTRA_DIR / split / oid / f"{oid}_combined_Total_DOS_Li_PDOS.csv"
            row = {
                "split": split,
                "ID": oid,
                "file": str(combined_path) if is_selected else "",
                "selected": is_selected,
                "candidate_rank": rank,
                "score": item["score"],
                "selection_margin": top_score - second_score if rank == 1 and np.isfinite(second_score) else np.nan,
                "source_total_dos": str(item["total_path"]),
                "source_li_pdos": str(item["li_path"]),
                "source_folder": str(item["folder"]),
            }
            if not is_selected:
                row["status"] = "not_selected_duplicate_candidate"
                rows.append(row)
                continue
            try:
                combine_total_and_li_csv(item["total_path"], item["li_path"], combined_path)
                columns = inspect_option_a_csv(combined_path)
                if columns is None:
                    raise ValueError("combined CSV did not produce Option-A columns")
                selected[oid] = (combined_path, columns)
                SPECTRUM_SOURCE_DIRS[str(combined_path.resolve())] = item["folder"]
                row.update({"status": "combined_from_separate_csv", **columns})
            except Exception as exc:
                row.update({"selected": False, "file": "", "status": "combine_failed", "error": str(exc)})
            rows.append(row)

    return selected, pd.DataFrame(rows)


train_spectra, train_spectrum_index = index_option_a_spectra(TRAIN_PDOS_ROOT, "train")
test_spectra, test_spectrum_index = index_option_a_spectra(TEST_PDOS_ROOT, "test")
spectrum_index = pd.concat([train_spectrum_index, test_spectrum_index], ignore_index=True)
spectrum_index.to_csv(TABLES / "option_A_spectrum_candidates.csv", index=False)

selected_spectra = {}
for split, collection in (("train", train_spectra), ("test", test_spectra)):
    for oid, (path, columns) in collection.items():
        selected_spectra[(split, oid)] = {"path": Path(path), "columns": columns}

# Apply spectrum overrides.
for oid, override in STRUCTURE_OVERRIDES.items():
    oid = normalize_id(oid)
    if override.get("spectrum"):
        for split in ("train", "test"):
            if (split, oid) in meta_by_id.index:
                path = Path(override["spectrum"])
                columns = inspect_option_a_csv(path)
                if columns is None:
                    raise ValueError(f"Spectrum override is not an Option-A CSV: {path}")
                selected_spectra[(split, oid)] = {"path": path, "columns": columns}

print(f"Option-A spectra found: {len(selected_spectra)}")

# -------------------------- INDEX AUXILIARY FILES ----------------------------
STRUCTURE_SUFFIXES = {".cif", ".vasp", ".poscar", ".contcar", ".yaml", ".yml", ".xz", ".gz"}

search_root_specs = [
    (TRAIN_PDOS_ROOT, "train", "production_root"),
    (TEST_PDOS_ROOT, "test", "production_root"),
]
for candidate in EXTRA_SEARCH_ROOTS:
    path = Path(candidate)
    if usable_directory(path):
        lower = str(path).lower()
        split_hint = "train" if "train" in lower else "test" if "test" in lower else None
        search_root_specs.append((path, split_hint, "extra_root"))

# Remove duplicate physical roots while preserving the first, most specific role.
unique_specs = []
seen_roots = set()
for root, split_hint, root_role in search_root_specs:
    key = str(Path(root).resolve())
    if key not in seen_roots:
        unique_specs.append((Path(root), split_hint, root_role))
        seen_roots.add(key)
search_root_specs = unique_specs
search_roots = [x[0] for x in search_root_specs]

structure_candidates = defaultdict(list)
eigen_candidates = defaultdict(list)
auxiliary_index_rows = []
seen_auxiliary_files = set()

for root, split_hint, root_role in search_root_specs:
    print(f"Indexing auxiliary files under: {root}")
    for path in root.rglob("*"):
        if not path.is_file():
            continue
        try:
            physical_key = str(path.resolve())
        except Exception:
            physical_key = str(path)
        if physical_key in seen_auxiliary_files:
            continue
        seen_auxiliary_files.add(physical_key)

        name_lower = path.name.lower()
        oid = extract_obelix_id(path, root, known_ids)
        if oid is None:
            continue
        candidate_key = (split_hint, oid) if split_hint is not None else (None, oid)
        if name_lower == "phonon_eigen_data.npz" or (
            path.suffix.lower() == ".npz" and "eigen" in name_lower
        ):
            eigen_candidates[candidate_key].append(path)
            auxiliary_index_rows.append({
                "split_hint": split_hint or "", "ID": oid, "root": str(root),
                "root_role": root_role, "kind": "eigen", "file": str(path)
            })
        elif (
            path.suffix.lower() in STRUCTURE_SUFFIXES
            or name_lower in {"poscar", "contcar"}
            or name_lower.endswith(("phonopy_params.yaml.xz", "phonopy.yaml.xz"))
        ):
            if any(term in name_lower for term in ["band", "mesh", "disp", "force_sets"]):
                continue
            structure_candidates[candidate_key].append(path)
            auxiliary_index_rows.append({
                "split_hint": split_hint or "", "ID": oid, "root": str(root),
                "root_role": root_role, "kind": "structure", "file": str(path)
            })

pd.DataFrame(auxiliary_index_rows).to_csv(TABLES / "indexed_auxiliary_files.csv", index=False)

# ------------------------------ STRUCTURE PARSING ----------------------------
def open_yaml_any(path):
    path = Path(path)
    name = path.name.lower()
    if name.endswith(".xz"):
        with lzma.open(path, "rt", errors="ignore") as handle:
            return yaml.safe_load(handle)
    if name.endswith(".gz"):
        with gzip.open(path, "rt", errors="ignore") as handle:
            return yaml.safe_load(handle)
    with open(path, "r", errors="ignore") as handle:
        return yaml.safe_load(handle)


def structure_from_phonopy_yaml(path):
    data = open_yaml_any(path)
    if not isinstance(data, dict):
        raise ValueError("YAML root is not a dictionary")
    blocks = []
    for key in ["unit_cell", "primitive_cell", "supercell"]:
        block = data.get(key)
        if isinstance(block, dict):
            blocks.append((key, block))
    # Some files nest a cell block under phonopy.
    if isinstance(data.get("phonopy"), dict):
        for key in ["unit_cell", "primitive_cell", "supercell"]:
            block = data["phonopy"].get(key)
            if isinstance(block, dict):
                blocks.append((f"phonopy/{key}", block))
    for label, block in blocks:
        lattice = block.get("lattice")
        points = block.get("points", block.get("atoms"))
        if lattice is None or not isinstance(points, list) or not points:
            continue
        species = []
        coords = []
        for point in points:
            if not isinstance(point, dict):
                break
            symbol = point.get("symbol", point.get("element", point.get("name")))
            coordinate = point.get("coordinates", point.get("position"))
            if symbol is None or coordinate is None:
                break
            species.append(str(symbol))
            coords.append(coordinate)
        if len(species) == len(points):
            return Structure(Lattice(lattice), species, coords), label
    raise ValueError("No unit_cell/primitive_cell block could be parsed")


def load_structure_any(path):
    path = Path(path)
    name = path.name.lower()
    errors = []
    if name.endswith((".yaml", ".yml", ".yaml.xz", ".yml.xz", ".yaml.gz", ".yml.gz")):
        try:
            structure, block = structure_from_phonopy_yaml(path)
            return structure, f"phonopy_yaml:{block}"
        except Exception as exc:
            errors.append(f"phonopy_yaml={exc}")
    try:
        return Structure.from_file(str(path)), "pymatgen_structure"
    except Exception as exc:
        errors.append(f"Structure.from_file={exc}")
    # CIFs with slightly non-normalized occupancies sometimes need a larger
    # parser occupancy tolerance.
    if path.suffix.lower() == ".cif":
        try:
            return Structure.from_file(str(path), occupancy_tolerance=2.0), "pymatgen_cif_tolerance2"
        except Exception as exc:
            errors.append(f"CIF tolerance2={exc}")
    raise ValueError("; ".join(errors))


def partial_occupancy_metrics(structure):
    partial_sites = 0
    mixed_sites = 0
    underoccupied_sites = 0
    overoccupied_sites = 0
    min_total_occupancy = np.inf
    max_total_occupancy = -np.inf
    for site in structure:
        occupancies = [float(v) for _, v in site.species.items()]
        total = float(sum(occupancies))
        min_total_occupancy = min(min_total_occupancy, total)
        max_total_occupancy = max(max_total_occupancy, total)
        if len(occupancies) > 1:
            mixed_sites += 1
        if len(occupancies) > 1 or any(v < 1 - 1e-8 for v in occupancies) or abs(total - 1) > 1e-8:
            partial_sites += 1
        if total < 1 - 1e-8:
            underoccupied_sites += 1
        if total > 1 + 1e-8:
            overoccupied_sites += 1
    if len(structure) == 0:
        min_total_occupancy = np.nan
        max_total_occupancy = np.nan
    return {
        "is_ordered": bool(structure.is_ordered),
        "partial_or_mixed_sites": partial_sites,
        "mixed_species_sites": mixed_sites,
        "underoccupied_sites": underoccupied_sites,
        "overoccupied_sites": overoccupied_sites,
        "min_site_total_occupancy": finite_or_nan(min_total_occupancy),
        "max_site_total_occupancy": finite_or_nan(max_total_occupancy),
    }


def minimum_periodic_distance(structure):
    if len(structure) < 2:
        return np.nan
    matrix = np.array(structure.distance_matrix, dtype=float)
    np.fill_diagonal(matrix, np.inf)
    value = np.min(matrix)
    return float(value) if np.isfinite(value) else np.nan


def structure_metrics(path):
    structure, parser = load_structure_any(path)
    counts = structure.composition.get_el_amt_dict()
    occupancy = partial_occupancy_metrics(structure)
    min_distance = minimum_periodic_distance(structure)
    return {
        "structure": structure,
        "parser": parser,
        "counts": {k: float(v) for k, v in counts.items()},
        "formula": structure.composition.formula,
        "reduced_formula": structure.composition.reduced_formula,
        "n_sites": len(structure),
        "volume_A3": float(structure.volume),
        "volume_per_site_A3": float(structure.volume / len(structure)) if len(structure) else np.nan,
        "density_g_cm3": float(structure.density),
        "min_distance_A": min_distance,
        **occupancy,
    }

# -------------------------- CANDIDATE SCORING -------------------------------
def _candidate_pool(mapping, split, oid):
    values = list(mapping.get((split, oid), [])) + list(mapping.get((None, oid), []))
    unique = []
    seen = set()
    for path in values:
        try:
            key = str(Path(path).resolve())
        except Exception:
            key = str(path)
        if key not in seen:
            unique.append(Path(path))
            seen.add(key)
    return unique


def structure_file_score(path, spectrum_path, role, oid):
    path = Path(path)
    name = path.name.lower()
    text = str(path).lower()
    score = 0.0
    spectrum_ref_dir = SPECTRUM_SOURCE_DIRS.get(
        str(Path(spectrum_path).resolve()), Path(spectrum_path).parent
    )
    distance = path_distance(path.parent, spectrum_ref_dir)
    same_folder = distance == 0
    score += max(0, 500 - 50 * distance)

    exact_relaxed = name == f"relaxed_{oid}.cif"
    exact_ordered = name == f"ordered_adaptive_{oid}.cif"

    if role == "production":
        if same_folder:
            score += 1500
        if exact_relaxed:
            score += 2500
        if name == "ultra_relaxed_generated.cif":
            score += 2200
        if "ultra_relaxed" in name or "final_relaxed" in name or "fully_relaxed" in name:
            score += 1600
        if "optimized" in name or "optimised" in name:
            score += 1000
        if "relaxed" in name:
            score += 700
        if "phonopy_params" in name:
            score += 900
        elif name.startswith("phonopy.yaml"):
            score += 800
        if exact_ordered or "ordered_adaptive" in name:
            score -= 1200
        if any(term in text for term in ["original", "raw_cif", "input_cif"]):
            score -= 1200
    else:  # source means the ordered pre-relaxation structure actually used.
        if same_folder:
            score += 1500
        if exact_ordered:
            score += 3000
        elif "ordered_adaptive" in name:
            score += 2200
        if any(term in name for term in ["original", "raw", "input", "starting"]):
            score += 400
        if "relaxed" in name or "ultra_relaxed" in name or "final_relaxed" in name:
            score -= 1800
        if name.endswith(".cif"):
            score += 150

    if any(term in text for term in ["supercell", "displaced", "disp-", "poscar-"]):
        score -= 3000
    if any(term in text for term in ["convergence", "reference_spectra", "nims"]):
        score -= 1000
    if any(term in text for term in ["backup", "archive", "old", "copy"]):
        score -= 500
    return score


def eigen_file_score(path, spectrum_path):
    path = Path(path)
    score = 500 if path.name.lower() == "phonon_eigen_data.npz" else 200
    spectrum_ref_dir = SPECTRUM_SOURCE_DIRS.get(
        str(Path(spectrum_path).resolve()), Path(spectrum_path).parent
    )
    distance = path_distance(path.parent, spectrum_ref_dir)
    score += max(0, 1000 - 100 * distance)
    if distance == 0:
        score += 1500
    text = str(path).lower()
    if "convergence" in text or "reference" in text:
        score -= 1000
    return score


def choose_files(split, oid, spectrum_path):
    override = STRUCTURE_OVERRIDES.get(oid, {})
    result = {}
    candidates = _candidate_pool(structure_candidates, split, oid)

    for role in ("production", "source"):
        if override.get(role):
            result[role] = Path(override[role])
            result[f"{role}_score"] = np.inf
            result[f"{role}_selection"] = "manual_override"
        else:
            ranked = sorted(
                [(structure_file_score(p, spectrum_path, role, oid), str(p), p) for p in candidates],
                reverse=True,
            )
            if role == "source" and result.get("production") is not None:
                ranked = [x for x in ranked if x[2].resolve() != result["production"].resolve()]
            minimum_score = (
                MIN_AUTOMATIC_PRODUCTION_SCORE if role == "production"
                else MIN_AUTOMATIC_SOURCE_SCORE
            )
            if ranked and ranked[0][0] >= minimum_score:
                result[role] = ranked[0][2]
                result[f"{role}_score"] = ranked[0][0]
                result[f"{role}_selection"] = (
                    "automatic_same_folder" if path_distance(
                        ranked[0][2].parent,
                        SPECTRUM_SOURCE_DIRS.get(str(Path(spectrum_path).resolve()), Path(spectrum_path).parent),
                    ) == 0 else "automatic_fallback"
                )
            elif ranked:
                result[role] = None
                result[f"{role}_score"] = ranked[0][0]
                result[f"{role}_selection"] = "ambiguous_below_score_threshold"
            else:
                result[role] = None
                result[f"{role}_score"] = np.nan
                result[f"{role}_selection"] = "missing"

    if override.get("eigen"):
        result["eigen"] = Path(override["eigen"])
        result["eigen_score"] = np.inf
        result["eigen_selection"] = "manual_override"
    else:
        ranked_eigen = sorted(
            [(eigen_file_score(p, spectrum_path), str(p), p)
             for p in _candidate_pool(eigen_candidates, split, oid)],
            reverse=True,
        )
        if ranked_eigen:
            result["eigen"] = ranked_eigen[0][2]
            result["eigen_score"] = ranked_eigen[0][0]
            result["eigen_selection"] = "automatic"
        else:
            result["eigen"] = None
            result["eigen_score"] = np.nan
            result["eigen_selection"] = "missing"

    source_path = result.get("source")
    if source_path is None:
        result["source_role"] = "missing"
    elif "ordered_adaptive" in Path(source_path).name.lower():
        result["source_role"] = "ordered_pre_relaxation_structure"
    else:
        result["source_role"] = "fallback_unverified_source_candidate"
    return result

# --------------------------- COMPOSITION COMPARISON --------------------------
def composition_counts(formula):
    composition = Composition(str(formula))
    return {k: float(v) for k, v in composition.get_el_amt_dict().items()}


def weighted_median(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    order = np.argsort(values)
    values = values[order]
    weights = weights[order]
    cutoff = 0.5 * np.sum(weights)
    return float(values[np.searchsorted(np.cumsum(weights), cutoff, side="left")])


def infer_integer_framework_scale(expected, actual):
    expected_elements = [e for e, v in expected.items() if v > 1e-12]
    framework = [e for e in expected_elements if e != "Li" and actual.get(e, 0.0) > 0]
    if not framework:
        framework = [e for e in expected_elements if actual.get(e, 0.0) > 0]
    if not framework:
        return np.nan, np.nan, []

    largest = max(expected[e] for e in framework)
    anchor_cutoff = max(ANCHOR_MIN_EXPECTED_COUNT, ANCHOR_MIN_FRACTION_OF_MAX * largest)
    anchors = [e for e in framework if expected[e] >= anchor_cutoff]
    if not anchors:
        anchors = framework

    ratios = [actual[e] / expected[e] for e in anchors]
    weights = [expected[e] for e in anchors]
    raw_scale = weighted_median(ratios, weights)
    integer_scale = float(max(1, int(round(raw_scale)))) if np.isfinite(raw_scale) else np.nan
    return raw_scale, integer_scale, anchors


def _hybrid_ok(abs_error, relative_error, abs_threshold, relative_threshold):
    return (
        (np.isfinite(abs_error) and abs_error <= abs_threshold + 1e-12)
        or (np.isfinite(relative_error) and relative_error <= relative_threshold + 1e-12)
    )


def compare_compositions(expected, actual):
    all_elements = sorted(set(expected) | set(actual))
    expected_elements = {e for e, v in expected.items() if v > 1e-10}
    actual_elements = {e for e, v in actual.items() if v > 1e-10}
    missing_elements = sorted(expected_elements - actual_elements)
    extra_elements = sorted(actual_elements - expected_elements)

    raw_scale, scale_framework, anchor_elements = infer_integer_framework_scale(expected, actual)
    if not np.isfinite(scale_framework):
        scale_framework = np.nan

    expected_scaled = {
        e: float(scale_framework * expected.get(e, 0.0)) if np.isfinite(scale_framework) else np.nan
        for e in all_elements
    }
    delta = {
        e: float(actual.get(e, 0.0) - expected_scaled.get(e, 0.0))
        if np.isfinite(expected_scaled.get(e, np.nan)) else np.nan
        for e in all_elements
    }
    absolute_errors = {e: abs(v) if np.isfinite(v) else np.nan for e, v in delta.items()}

    relative_errors = {}
    for e in expected_elements:
        denominator = abs(expected_scaled.get(e, np.nan))
        relative_errors[e] = (
            abs(delta[e]) / denominator
            if np.isfinite(denominator) and denominator > 1e-12 else np.nan
        )

    framework_elements = sorted(e for e in expected_elements if e != "Li")
    max_relative = max((v for v in relative_errors.values() if np.isfinite(v)), default=np.nan)
    framework_max_relative = max(
        (relative_errors[e] for e in framework_elements if np.isfinite(relative_errors.get(e, np.nan))),
        default=np.nan,
    )
    max_absolute = max((v for v in absolute_errors.values() if np.isfinite(v)), default=np.nan)
    framework_max_absolute = max(
        (absolute_errors[e] for e in framework_elements if np.isfinite(absolute_errors.get(e, np.nan))),
        default=np.nan,
    )

    e_total = sum(expected_scaled.get(e, 0.0) for e in all_elements)
    a_total = sum(actual.get(e, 0.0) for e in all_elements)
    if e_total > 0 and a_total > 0:
        expected_fraction = {e: expected_scaled.get(e, 0.0) / e_total for e in all_elements}
        actual_fraction = {e: actual.get(e, 0.0) / a_total for e in all_elements}
        total_variation = 0.5 * sum(
            abs(expected_fraction[e] - actual_fraction[e]) for e in all_elements
        )
    else:
        total_variation = np.nan

    expected_li = expected_scaled.get("Li", 0.0)
    actual_li = actual.get("Li", 0.0)
    li_delta = actual_li - expected_li if np.isfinite(expected_li) else np.nan
    li_absolute = abs(li_delta) if np.isfinite(li_delta) else np.nan
    li_relative_signed = li_delta / expected_li if np.isfinite(expected_li) and expected_li > 1e-12 else np.nan
    li_relative_abs = abs(li_relative_signed) if np.isfinite(li_relative_signed) else np.nan

    pass_flags = {}
    warn_flags = {}
    for e in expected_elements:
        pass_flags[e] = _hybrid_ok(
            absolute_errors[e], relative_errors[e],
            PASS_MAX_ABSOLUTE_COUNT_ERROR_ATOMS, PASS_MAX_RELATIVE_STOICH_ERROR,
        )
        warn_flags[e] = _hybrid_ok(
            absolute_errors[e], relative_errors[e],
            WARN_MAX_ABSOLUTE_COUNT_ERROR_ATOMS, WARN_MAX_RELATIVE_STOICH_ERROR,
        )

    framework_pass = all(pass_flags.get(e, False) for e in framework_elements)
    framework_warn = all(warn_flags.get(e, False) for e in framework_elements)
    li_pass = (
        "Li" not in expected_elements
        or _hybrid_ok(
            li_absolute, li_relative_abs,
            LI_PASS_MAX_ABSOLUTE_COUNT_ERROR_ATOMS, PASS_MAX_RELATIVE_STOICH_ERROR,
        )
    )
    li_warn = (
        "Li" not in expected_elements
        or _hybrid_ok(
            li_absolute, li_relative_abs,
            LI_WARN_MAX_ABSOLUTE_COUNT_ERROR_ATOMS, WARN_MAX_RELATIVE_STOICH_ERROR,
        )
    )

    exact_proportional = (
        not missing_elements and not extra_elements
        and all(np.isfinite(delta[e]) and abs(delta[e]) <= 1e-8 for e in expected_elements)
    )

    if missing_elements or extra_elements:
        status = "FAIL_ELEMENT_SET"
    elif framework_warn and not li_warn:
        status = "FAIL_LI_STOICHIOMETRY"
    elif framework_pass and li_pass and np.isfinite(total_variation) and total_variation <= PASS_TOTAL_VARIATION_ERROR:
        status = "PASS" if exact_proportional else "PASS_INTEGERIZED_COMPOSITION"
    elif framework_warn and li_warn and np.isfinite(total_variation) and total_variation <= WARN_TOTAL_VARIATION_ERROR:
        status = "WARN_INTEGERIZATION_APPROXIMATION"
    else:
        status = "FAIL_GENERAL_STOICHIOMETRY"

    n_outside_pass = sum(not pass_flags.get(e, False) for e in expected_elements)
    n_outside_warn = sum(not warn_flags.get(e, False) for e in expected_elements)
    scale_nearest_integer = round(raw_scale) if np.isfinite(raw_scale) else np.nan
    scale_integer_deviation = (
        abs(raw_scale - scale_nearest_integer) if np.isfinite(raw_scale) else np.nan
    )

    return {
        "composition_status": status,
        "scale_all": scale_framework,
        "framework_scale_raw": raw_scale,
        "framework_scale_formula_units": scale_framework,
        "framework_scale_nearest_integer": scale_nearest_integer,
        "framework_scale_integer_deviation": scale_integer_deviation,
        "framework_scale_anchor_elements": ";".join(anchor_elements),
        "max_relative_stoich_error": max_relative,
        "framework_max_relative_stoich_error": framework_max_relative,
        "max_absolute_count_error_atoms": max_absolute,
        "framework_max_absolute_count_error_atoms": framework_max_absolute,
        "n_elements_outside_pass_tolerance": n_outside_pass,
        "n_elements_outside_warn_tolerance": n_outside_warn,
        "composition_total_variation_error": total_variation,
        "expected_Li_at_framework_scale": expected_li,
        "actual_Li": actual_li,
        "Li_count_error": li_delta,
        "Li_absolute_count_error": li_absolute,
        "Li_relative_error": li_relative_signed,
        "missing_elements": ";".join(missing_elements),
        "extra_elements": ";".join(extra_elements),
        "expected_counts_at_framework_scale_json": json_counts(expected_scaled),
        "actual_counts_json": json_counts(actual),
        "actual_minus_expected_json": json_counts(delta),
        "element_absolute_errors_json": json.dumps(
            {k: round(float(v), 8) for k, v in sorted(absolute_errors.items()) if np.isfinite(v)},
            sort_keys=True,
        ),
        "element_relative_errors_json": json.dumps(
            {k: round(float(v), 8) for k, v in sorted(relative_errors.items()) if np.isfinite(v)},
            sort_keys=True,
        ),
        "element_pass_flags_json": json.dumps(pass_flags, sort_keys=True),
        "element_warn_flags_json": json.dumps(warn_flags, sort_keys=True),
    }


def oxidation_state_heuristic(counts):
    if not RUN_OXIDATION_STATE_HEURISTIC:
        return {"neutral_oxidation_guess_status": "not_run", "neutral_oxidation_guess_json": ""}
    try:
        comp = Composition(counts).reduced_composition
        if len(comp.elements) > MAX_ELEMENTS_FOR_OXIDATION_GUESS:
            return {
                "neutral_oxidation_guess_status": "skipped_too_many_elements",
                "neutral_oxidation_guess_json": "",
            }
        guesses = comp.oxi_state_guesses(max_sites=100)
        if guesses:
            return {
                "neutral_oxidation_guess_status": "neutral_assignment_found",
                "neutral_oxidation_guess_json": json.dumps(guesses[0], sort_keys=True),
            }
        return {"neutral_oxidation_guess_status": "no_neutral_assignment_found", "neutral_oxidation_guess_json": ""}
    except Exception as exc:
        return {
            "neutral_oxidation_guess_status": "error",
            "neutral_oxidation_guess_json": str(exc)[:500],
        }

# ------------------------------- SPECTRAL QC --------------------------------
def read_option_a_spectrum(path, columns):
    frame = pd.read_csv(path)
    freq = pd.to_numeric(frame[columns["frequency"]], errors="coerce").to_numpy(float)
    total = pd.to_numeric(frame[columns["total"]], errors="coerce").to_numpy(float)
    li = pd.to_numeric(frame[columns["li"]], errors="coerce").to_numpy(float)
    finite = np.isfinite(freq)
    freq = freq[finite]
    total = total[finite]
    li = li[finite]
    order = np.argsort(freq)
    return freq[order], total[order], li[order]


def spectrum_qc(path, columns):
    freq, total, li = read_option_a_spectrum(path, columns)
    unique_freq = np.unique(freq)
    differences = np.diff(unique_freq)
    positive_differences = differences[differences > 0]
    median_spacing = float(np.median(positive_differences)) if len(positive_differences) else np.nan
    max_spacing = float(np.max(positive_differences)) if len(positive_differences) else np.nan
    positive_mask = (freq > 0) & (freq <= 100)
    total_positive_area = float(np.trapezoid(np.clip(total[positive_mask], 0, None), freq[positive_mask])) if np.sum(positive_mask) >= 2 else np.nan
    li_positive_area = float(np.trapezoid(np.clip(li[positive_mask], 0, None), freq[positive_mask])) if np.sum(positive_mask) >= 2 else np.nan
    return {
        "n_frequency_points": len(freq),
        "min_plotted_frequency_THz": float(np.min(freq)) if len(freq) else np.nan,
        "max_plotted_frequency_THz": float(np.max(freq)) if len(freq) else np.nan,
        "median_native_spacing_THz": median_spacing,
        "max_native_spacing_THz": max_spacing,
        "points_0_to_100_THz": int(np.sum(positive_mask)),
        "total_positive_area_0_to_100": total_positive_area,
        "Li_positive_area_0_to_100": li_positive_area,
        "coarse_grid_flag": bool(np.isfinite(median_spacing) and median_spacing > COARSE_DOS_GRID_THz),
        "high_max_frequency_flag": bool(len(freq) and np.max(freq) > HIGH_MAX_FREQUENCY_THz),
        "very_high_max_frequency_flag": bool(len(freq) and np.max(freq) > VERY_HIGH_MAX_FREQUENCY_THz),
        "negative_plotted_grid_endpoint_flag": bool(len(freq) and np.min(freq) < 0),
    }


def eigen_qc(path):
    with np.load(path, allow_pickle=False) as data:
        if "frequencies" not in data.files:
            raise KeyError("NPZ does not contain a 'frequencies' array")
        frequencies = np.asarray(data["frequencies"], dtype=float)
        weights = np.asarray(data["weights"], dtype=float) if "weights" in data.files else None
        q_points = np.asarray(data["q_points"], dtype=float) if "q_points" in data.files else None
    finite = frequencies[np.isfinite(frequencies)]
    if finite.size == 0:
        raise ValueError("No finite mesh frequencies")
    output = {
        "mesh_frequency_shape": str(tuple(frequencies.shape)),
        "n_q_points": int(frequencies.shape[0]) if frequencies.ndim >= 2 else np.nan,
        "n_branches": int(frequencies.shape[-1]) if frequencies.ndim >= 2 else np.nan,
        "true_min_eigenfrequency_THz": float(np.min(finite)),
        "true_max_eigenfrequency_THz": float(np.max(finite)),
        "q_points_shape": str(tuple(q_points.shape)) if q_points is not None else "",
    }
    for threshold in IMAGINARY_THRESHOLDS_THz:
        label = str(abs(threshold)).replace(".", "p")
        raw_fraction = float(np.mean(finite < threshold))
        output[f"fraction_modes_below_minus_{label}THz"] = raw_fraction
        output[f"count_modes_below_minus_{label}THz"] = int(np.sum(finite < threshold))
        if weights is not None and frequencies.ndim == 2 and len(weights) == frequencies.shape[0]:
            mask = frequencies < threshold
            weighted_count = np.sum(weights[:, None] * mask)
            denominator = np.sum(weights) * frequencies.shape[1]
            output[f"weighted_fraction_modes_below_minus_{label}THz"] = float(weighted_count / denominator)
        else:
            output[f"weighted_fraction_modes_below_minus_{label}THz"] = np.nan
    return output

# ------------------------------ RUN FULL AUDIT -------------------------------
audit_rows = []
structure_candidate_rows = []
source_ordering_rows = []
spectrum_rows = []
eigen_rows = []
missing_rows = []
error_rows = []

for counter, ((split, oid), spectrum_info) in enumerate(sorted(selected_spectra.items()), start=1):
    spectrum_path = spectrum_info["path"]
    columns = spectrum_info["columns"]
    print(f"[{counter:3d}/{len(selected_spectra)}] {split}/{oid}")

    if (split, oid) not in meta_by_id.index:
        missing_rows.append({
            "split": split, "ID": oid, "issue": "ID_not_found_in_official_metadata",
            "file": str(spectrum_path),
        })
        continue
    meta = meta_by_id.loc[(split, oid)]
    reduced_value = meta.get("Reduced Composition", "")
    composition_value = meta.get("Composition", "")
    intended_formula = str(reduced_value).strip()
    if intended_formula.lower() in {"", "nan", "none"}:
        intended_formula = str(composition_value).strip()
    family = str(meta.get("Family", "Unknown"))
    try:
        expected_counts = composition_counts(intended_formula)
    except Exception as exc:
        error_rows.append({
            "split": split, "ID": oid, "stage": "parse_expected_formula",
            "file": "", "error": str(exc),
        })
        continue

    selected = choose_files(split, oid, spectrum_path)

    # Record all structure candidates and their parseable compositions.
    ranked_production = sorted(
        [(structure_file_score(p, spectrum_path, "production", oid), str(p), p) for p in _candidate_pool(structure_candidates, split, oid)],
        reverse=True,
    )[:MAX_STRUCTURE_CANDIDATES_PER_ID]
    for rank, (score, _, candidate_path) in enumerate(ranked_production, start=1):
        candidate_row = {
            "split": split, "ID": oid, "family": family,
            "candidate_rank_for_production": rank,
            "production_score": score,
            "source_score": structure_file_score(candidate_path, spectrum_path, "source", oid),
            "selected_as_production": selected["production"] is not None and candidate_path.resolve() == selected["production"].resolve(),
            "selected_as_source": selected["source"] is not None and candidate_path.resolve() == selected["source"].resolve(),
            "file": str(candidate_path),
        }
        try:
            metrics = structure_metrics(candidate_path)
            comparison = compare_compositions(expected_counts, metrics["counts"])
            candidate_row.update({
                "parse_status": "ok",
                "parser": metrics["parser"],
                "formula": metrics["formula"],
                "reduced_formula": metrics["reduced_formula"],
                "counts_json": json_counts(metrics["counts"]),
                "n_sites": metrics["n_sites"],
                "is_ordered": metrics["is_ordered"],
                "partial_or_mixed_sites": metrics["partial_or_mixed_sites"],
                "comparison_to_OBELiX": comparison["composition_status"],
                "Li_relative_error": comparison["Li_relative_error"],
                "composition_total_variation_error": comparison["composition_total_variation_error"],
            })
        except Exception as exc:
            candidate_row.update({"parse_status": "failed", "error": str(exc)[:1000]})
        structure_candidate_rows.append(candidate_row)

    # Parse production structure.
    production_metrics = None
    production_comparison = None
    if selected["production"] is not None and selected["production"].exists():
        try:
            production_metrics = structure_metrics(selected["production"])
            production_comparison = compare_compositions(expected_counts, production_metrics["counts"])
        except Exception as exc:
            error_rows.append({
                "split": split, "ID": oid, "stage": "production_structure_parse",
                "file": str(selected["production"]), "error": str(exc),
            })
    else:
        missing_rows.append({
            "split": split, "ID": oid, "issue": "missing_production_structure",
            "file": str(spectrum_path),
        })

    # Parse source structure, if distinct and available.
    source_metrics = None
    source_comparison = None
    if selected["source"] is not None and selected["source"].exists():
        try:
            source_metrics = structure_metrics(selected["source"])
            source_comparison = compare_compositions(expected_counts, source_metrics["counts"])
        except Exception as exc:
            error_rows.append({
                "split": split, "ID": oid, "stage": "source_structure_parse",
                "file": str(selected["source"]), "error": str(exc),
            })

    # Spectrum grid QC.
    try:
        sqc = spectrum_qc(spectrum_path, columns)
        spectrum_rows.append({
            "split": split, "ID": oid, "family": family,
            "spectrum_file": str(spectrum_path), **sqc,
        })
    except Exception as exc:
        sqc = {}
        error_rows.append({
            "split": split, "ID": oid, "stage": "spectrum_qc",
            "file": str(spectrum_path), "error": str(exc),
        })

    # True mesh-eigenfrequency QC.
    eqc = {}
    if selected["eigen"] is not None and selected["eigen"].exists():
        try:
            eqc = eigen_qc(selected["eigen"])
            eigen_rows.append({
                "split": split, "ID": oid, "family": family,
                "eigen_file": str(selected["eigen"]), **eqc,
            })
        except Exception as exc:
            error_rows.append({
                "split": split, "ID": oid, "stage": "eigen_qc",
                "file": str(selected["eigen"]), "error": str(exc),
            })
    else:
        missing_rows.append({
            "split": split, "ID": oid, "issue": "missing_phonon_eigen_data_npz",
            "file": str(spectrum_path),
        })

    # Main audit row.
    row = {
        "split": split,
        "ID": oid,
        "family": family,
        "intended_formula_OBELiX": intended_formula,
        "expected_formula_counts_json": json_counts(expected_counts),
        "spectrum_file": str(spectrum_path),
        "production_structure_file": str(selected["production"]) if selected["production"] else "",
        "production_structure_selection": selected["production_selection"],
        "production_structure_score": selected["production_score"],
        "source_structure_file": str(selected["source"]) if selected["source"] else "",
        "source_structure_selection": selected["source_selection"],
        "source_structure_score": selected["source_score"],
        "source_structure_role": selected.get("source_role", ""),
        "eigen_file": str(selected["eigen"]) if selected["eigen"] else "",
        "eigen_selection": selected["eigen_selection"],
        "eigen_score": selected["eigen_score"],
    }

    if production_metrics is None or production_comparison is None:
        row.update({
            "audit_status": "MISSING_OR_UNREADABLE_PRODUCTION_STRUCTURE",
            "production_formula": "",
            "production_reduced_formula": "",
        })
    else:
        min_distance = production_metrics["min_distance_A"]
        geometry_status = (
            "FAIL_SEVERE_OVERLAP" if np.isfinite(min_distance) and min_distance < SEVERE_MIN_DISTANCE_A
            else "WARN_SHORT_DISTANCE" if np.isfinite(min_distance) and min_distance < WARNING_MIN_DISTANCE_A
            else "PASS"
        )
        charge_info = oxidation_state_heuristic(production_metrics["counts"])
        audit_status = production_comparison["composition_status"]
        if audit_status in {"PASS", "PASS_INTEGERIZED_COMPOSITION"} and geometry_status != "PASS":
            audit_status = geometry_status
        row.update({
            "audit_status": audit_status,
            "production_parser": production_metrics["parser"],
            "production_formula": production_metrics["formula"],
            "production_reduced_formula": production_metrics["reduced_formula"],
            "production_n_sites": production_metrics["n_sites"],
            "production_volume_A3": production_metrics["volume_A3"],
            "production_volume_per_site_A3": production_metrics["volume_per_site_A3"],
            "production_density_g_cm3": production_metrics["density_g_cm3"],
            "production_min_distance_A": min_distance,
            "geometry_status": geometry_status,
            "production_is_ordered": production_metrics["is_ordered"],
            "production_partial_or_mixed_sites": production_metrics["partial_or_mixed_sites"],
            "production_mixed_species_sites": production_metrics["mixed_species_sites"],
            "production_underoccupied_sites": production_metrics["underoccupied_sites"],
            "production_overoccupied_sites": production_metrics["overoccupied_sites"],
            **production_comparison,
            **charge_info,
        })

    if source_metrics is not None and source_comparison is not None:
        row.update({
            "source_parser": source_metrics["parser"],
            "source_formula": source_metrics["formula"],
            "source_reduced_formula": source_metrics["reduced_formula"],
            "source_counts_json": json_counts(source_metrics["counts"]),
            "source_is_ordered": source_metrics["is_ordered"],
            "source_partial_or_mixed_sites": source_metrics["partial_or_mixed_sites"],
            "source_mixed_species_sites": source_metrics["mixed_species_sites"],
            "source_underoccupied_sites": source_metrics["underoccupied_sites"],
            "source_comparison_to_OBELiX": source_comparison["composition_status"],
            "source_Li_relative_error_to_OBELiX": source_comparison["Li_relative_error"],
        })
    else:
        row.update({
            "source_formula": "", "source_counts_json": "",
            "source_is_ordered": np.nan, "source_partial_or_mixed_sites": np.nan,
        })

    row.update({f"spectrum_{k}": v for k, v in sqc.items()})
    row.update({f"eigen_{k}": v for k, v in eqc.items()})
    audit_rows.append(row)

    # Ordered pre-relaxation to relaxed-production count-preservation audit.
    if source_metrics is not None and production_metrics is not None:
        source_counts = source_metrics["counts"]
        production_counts = production_metrics["counts"]
        all_source_elements = sorted(set(source_counts) | set(production_counts))
        count_delta = {
            e: float(production_counts.get(e, 0.0) - source_counts.get(e, 0.0))
            for e in all_source_elements
        }
        max_abs_delta = max((abs(v) for v in count_delta.values()), default=0.0)
        element_set_preserved = {
            e for e, v in source_counts.items() if v > 1e-10
        } == {
            e for e, v in production_counts.items() if v > 1e-10
        }
        exact_count_preservation = element_set_preserved and max_abs_delta <= 1e-8
        if selected.get("source_role") != "ordered_pre_relaxation_structure":
            workflow_status = "UNVERIFIED_SOURCE_ROLE"
        else:
            workflow_status = (
                "PASS_EXACT_COUNT_PRESERVATION"
                if exact_count_preservation else "FAIL_RELAXATION_CHANGED_COMPOSITION"
            )
        source_to_obelix = compare_compositions(expected_counts, source_counts)
        production_to_obelix = compare_compositions(expected_counts, production_counts)
        source_ordering_rows.append({
            "split": split,
            "ID": oid,
            "family": family,
            "intended_formula_OBELiX": intended_formula,
            "source_role": selected.get("source_role", ""),
            "source_file": str(selected["source"]),
            "production_file": str(selected["production"]),
            "source_formula": source_metrics["formula"],
            "production_formula": production_metrics["formula"],
            "source_counts_json": json_counts(source_counts),
            "production_counts_json": json_counts(production_counts),
            "source_partial_or_mixed_sites": source_metrics["partial_or_mixed_sites"],
            "source_mixed_species_sites": source_metrics["mixed_species_sites"],
            "source_underoccupied_sites": source_metrics["underoccupied_sites"],
            "source_to_production_status": workflow_status,
            "source_to_OBELiX_status": source_to_obelix["composition_status"],
            "production_to_OBELiX_status": production_to_obelix["composition_status"],
            "source_to_production_max_absolute_count_change": max_abs_delta,
            "source_to_production_element_set_preserved": element_set_preserved,
            "source_to_production_Li_count_error": count_delta.get("Li", 0.0),
            "vacancy_or_partial_occupancy_preservation": (
                "source_role_unverified"
                if selected.get("source_role") != "ordered_pre_relaxation_structure"
                else "exactly_preserved_through_relaxation"
                if exact_count_preservation else "not_preserved_through_relaxation"
            ),
            "source_to_production_delta_json": json_counts(count_delta),
        })

# ----------------------------- SAVE PRIMARY TABLES ---------------------------
def table_from_rows(rows, columns=None, sort_cols=("split", "ID")):
    frame = pd.DataFrame(rows)
    if frame.empty and columns is not None:
        frame = pd.DataFrame(columns=columns)
    if sort_cols and all(col in frame.columns for col in sort_cols):
        frame = frame.sort_values(list(sort_cols)).reset_index(drop=True)
    else:
        frame = frame.reset_index(drop=True)
    return frame


audit_df = table_from_rows(audit_rows, columns=["split", "ID"])
structure_candidates_df = table_from_rows(structure_candidate_rows)
source_ordering_df = table_from_rows(source_ordering_rows, columns=["split", "ID"])
spectrum_df = table_from_rows(spectrum_rows, columns=["split", "ID"])
eigen_df = table_from_rows(
    eigen_rows,
    columns=[
        "split",
        "ID",
        "eigen_file",
        "true_min_eigenfrequency_THz",
        "true_max_eigenfrequency_THz",
        "fraction_modes_below_minus_0p1THz",
        "fraction_modes_below_minus_0p5THz",
        "fraction_modes_below_minus_1p0THz",
    ],
)
missing_df = table_from_rows(missing_rows, columns=["split", "ID"])
errors_df = table_from_rows(error_rows, columns=["split", "ID"])

primary_files = {
    "production_structure_composition_audit.csv": audit_df,
    "all_structure_candidates.csv": structure_candidates_df,
    "source_to_production_ordering_audit.csv": source_ordering_df,
    "spectrum_grid_qc.csv": spectrum_df,
    "eigenfrequency_stability_qc.csv": eigen_df,
    "missing_or_ambiguous_files.csv": missing_df,
    "audit_errors.csv": errors_df,
}
for filename, frame in primary_files.items():
    frame.to_csv(TABLES / filename, index=False)

# ------------------------- REGENERATION PRIORITY TABLE -----------------------
def severity_rank(status):
    ranking = {
        "FAIL_ELEMENT_SET": 1,
        "FAIL_LI_STOICHIOMETRY": 2,
        "FAIL_GENERAL_STOICHIOMETRY": 3,
        "MISSING_OR_UNREADABLE_PRODUCTION_STRUCTURE": 4,
        "FAIL_SEVERE_OVERLAP": 5,
        "WARN_SHORT_DISTANCE": 6,
        "WARN_INTEGERIZATION_APPROXIMATION": 7,
        "PASS_INTEGERIZED_COMPOSITION": 98,
        "PASS": 99,
    }
    return ranking.get(str(status), 50)

priority_rows = []
for _, row in audit_df.iterrows():
    reasons = []
    status = row.get("audit_status", "")
    if str(status).startswith("FAIL") or str(status).startswith("MISSING"):
        reasons.append(str(status))
    elif str(status).startswith("WARN"):
        reasons.append(str(status))
    coarse_value = row.get("spectrum_coarse_grid_flag", False)
    high_value = row.get("spectrum_very_high_max_frequency_flag", False)
    if pd.notna(coarse_value) and bool(coarse_value):
        reasons.append("COARSE_DOS_GRID")
    if pd.notna(high_value) and bool(high_value):
        reasons.append("VERY_HIGH_MAX_FREQUENCY")
    min_eigen = row.get("eigen_true_min_eigenfrequency_THz", np.nan)
    if np.isfinite(finite_or_nan(min_eigen)) and min_eigen < -0.5:
        reasons.append("TRUE_IMAGINARY_MODES_BELOW_-0.5THz")
    if row.get("eigen_file", "") == "":
        reasons.append("MISSING_EIGEN_FREQUENCIES")
    if reasons:
        priority_rows.append({
            "split": row["split"],
            "ID": row["ID"],
            "family": row.get("family", ""),
            "intended_formula_OBELiX": row.get("intended_formula_OBELiX", ""),
            "production_formula": row.get("production_formula", ""),
            "audit_status": status,
            "priority_rank": severity_rank(status),
            "reasons": ";".join(reasons),
            "Li_relative_error": row.get("Li_relative_error", np.nan),
            "composition_total_variation_error": row.get("composition_total_variation_error", np.nan),
            "median_native_spacing_THz": row.get("spectrum_median_native_spacing_THz", np.nan),
            "max_plotted_frequency_THz": row.get("spectrum_max_plotted_frequency_THz", np.nan),
            "true_min_eigenfrequency_THz": row.get("eigen_true_min_eigenfrequency_THz", np.nan),
            "production_structure_file": row.get("production_structure_file", ""),
            "spectrum_file": row.get("spectrum_file", ""),
        })
priority_df = pd.DataFrame(priority_rows)
if not priority_df.empty:
    priority_df = priority_df.sort_values(
        ["priority_rank", "split", "ID"], na_position="last"
    ).reset_index(drop=True)
priority_df.to_csv(TABLES / "regeneration_priority.csv", index=False)

# ------------------------------- SUMMARY TABLES ------------------------------
summary_rows = []
summary_rows.append({"metric": "Option-A spectra audited", "value": len(audit_df)})
summary_rows.append({"metric": "Train spectra audited", "value": int((audit_df["split"] == "train").sum())})
summary_rows.append({"metric": "Test spectra audited", "value": int((audit_df["split"] == "test").sum())})
for status, count in audit_df.get("audit_status", pd.Series(dtype=str)).value_counts(dropna=False).items():
    summary_rows.append({"metric": f"audit_status::{status}", "value": int(count)})
if "spectrum_coarse_grid_flag" in audit_df.columns:
    summary_rows.append({
        "metric": f"coarse DOS grid > {COARSE_DOS_GRID_THz:g} THz",
        "value": int(audit_df["spectrum_coarse_grid_flag"].fillna(False).sum()),
    })
if "spectrum_very_high_max_frequency_flag" in audit_df.columns:
    summary_rows.append({
        "metric": f"max plotted frequency > {VERY_HIGH_MAX_FREQUENCY_THz:g} THz",
        "value": int(audit_df["spectrum_very_high_max_frequency_flag"].fillna(False).sum()),
    })
if "eigen_true_min_eigenfrequency_THz" in audit_df.columns:
    min_eigen = pd.to_numeric(audit_df["eigen_true_min_eigenfrequency_THz"], errors="coerce")
    summary_rows.append({"metric": "structures with true min eigenfrequency < -0.1 THz", "value": int((min_eigen < -0.1).sum())})
    summary_rows.append({"metric": "structures with true min eigenfrequency < -0.5 THz", "value": int((min_eigen < -0.5).sum())})
    summary_rows.append({"metric": "structures with true min eigenfrequency < -1.0 THz", "value": int((min_eigen < -1.0).sum())})
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(TABLES / "audit_summary.csv", index=False)

status_by_split = pd.crosstab(audit_df["audit_status"], audit_df["split"], margins=True)
status_by_split.to_csv(TABLES / "audit_status_by_split.csv")

if "composition_status" in audit_df.columns:
    composition_status_by_split = pd.crosstab(
        audit_df["composition_status"], audit_df["split"], margins=True
    )
    composition_status_by_split.to_csv(TABLES / "composition_status_by_split.csv")

# --------------------------- PUBLICATION-LEVEL FIGURES -----------------------
def save_figure(fig, stem):
    fig.savefig(FIGURES / f"{stem}.pdf", bbox_inches="tight")
    fig.savefig(FIGURES / f"{stem}.png", dpi=FIGURE_DPI, bbox_inches="tight")
    plt.close(fig)

# Figure 1: audit status counts by split.
if not audit_df.empty:
    count_table = pd.crosstab(audit_df["audit_status"], audit_df["split"])
    count_table = count_table.sort_values(by=list(count_table.columns), ascending=False)
    fig, ax = plt.subplots(figsize=(9, max(4.5, 0.55 * len(count_table))))
    count_table.plot(kind="barh", ax=ax)
    ax.set_xlabel("Number of production spectra")
    ax.set_ylabel("Composition/geometry audit status")
    ax.set_title("OBELiX production-structure audit")
    ax.grid(axis="x", alpha=0.25)
    ax.legend(title="Official split")
    save_figure(fig, "01_audit_status_by_split")

# Figure 2: expected versus actual Li counts after framework scaling.
if {"expected_Li_at_framework_scale", "actual_Li"}.issubset(audit_df.columns):
    x = pd.to_numeric(audit_df["expected_Li_at_framework_scale"], errors="coerce")
    y = pd.to_numeric(audit_df["actual_Li"], errors="coerce")
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.any():
        fig, ax = plt.subplots(figsize=(6.5, 6.0))
        for split in sorted(audit_df.loc[valid, "split"].unique()):
            mask = valid & audit_df["split"].eq(split)
            ax.scatter(x[mask], y[mask], label=split, alpha=0.75, s=30)
        bound = max(float(np.nanmax(x[valid])), float(np.nanmax(y[valid])))
        ax.plot([0, bound], [0, bound], linestyle="--", linewidth=1.2, label="exact Li stoichiometry")
        ax.set_xlabel("Expected Li count from non-Li framework scaling")
        ax.set_ylabel("Actual Li count in production structure")
        ax.set_title("Audit of Li/vacancy preservation")
        ax.grid(alpha=0.25)
        ax.legend()
        save_figure(fig, "02_expected_vs_actual_Li_counts")

# Figure 3: largest framework-scaled Li count errors.
if "Li_count_error" in audit_df.columns:
    temp = audit_df.copy()
    temp["Li_count_error_numeric"] = pd.to_numeric(temp["Li_count_error"], errors="coerce")
    temp = temp[np.isfinite(temp["Li_count_error_numeric"])].copy()
    temp["abs_error"] = temp["Li_count_error_numeric"].abs()
    temp = temp.sort_values("abs_error", ascending=False).head(30).sort_values("Li_count_error_numeric")
    if not temp.empty:
        fig, ax = plt.subplots(figsize=(9, max(5, 0.28 * len(temp))))
        labels = temp["split"].str[0].str.upper() + ":" + temp["ID"].astype(str)
        ax.barh(labels, temp["Li_count_error_numeric"])
        ax.axvline(0, linewidth=1)
        ax.axvline(LI_PASS_MAX_ABSOLUTE_COUNT_ERROR_ATOMS, linestyle="--", linewidth=1, label="pass rounding bound")
        ax.axvline(-LI_PASS_MAX_ABSOLUTE_COUNT_ERROR_ATOMS, linestyle="--", linewidth=1)
        ax.axvline(LI_WARN_MAX_ABSOLUTE_COUNT_ERROR_ATOMS, linestyle=":", linewidth=1, label="warning bound")
        ax.axvline(-LI_WARN_MAX_ABSOLUTE_COUNT_ERROR_ATOMS, linestyle=":", linewidth=1)
        ax.set_xlabel("Framework-scaled Li count error (atoms in production cell)")
        ax.set_ylabel("OBELiX ID")
        ax.set_title("Largest Li/vacancy count deviations")
        ax.grid(axis="x", alpha=0.25)
        ax.legend()
        save_figure(fig, "03_largest_Li_stoichiometry_errors")

# Figure 4: DOS-grid spacing versus maximum plotted frequency.
if {"spectrum_median_native_spacing_THz", "spectrum_max_plotted_frequency_THz"}.issubset(audit_df.columns):
    x = pd.to_numeric(audit_df["spectrum_median_native_spacing_THz"], errors="coerce")
    y = pd.to_numeric(audit_df["spectrum_max_plotted_frequency_THz"], errors="coerce")
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.any():
        fig, ax = plt.subplots(figsize=(7.0, 5.5))
        for split in sorted(audit_df.loc[valid, "split"].unique()):
            mask = valid & audit_df["split"].eq(split)
            ax.scatter(x[mask], y[mask], label=split, alpha=0.75, s=30)
        ax.axvline(COARSE_DOS_GRID_THz, linestyle="--", linewidth=1.2)
        ax.axhline(HIGH_MAX_FREQUENCY_THz, linestyle="--", linewidth=1.2)
        ax.set_xlabel("Median native DOS-grid spacing (THz)")
        ax.set_ylabel("Maximum plotted frequency (THz)")
        ax.set_title("Spectral-grid quality control")
        ax.grid(alpha=0.25)
        ax.legend()
        save_figure(fig, "04_DOS_grid_spacing_vs_max_frequency")

# Figure 5: true minimum mesh eigenfrequency distribution.
if "eigen_true_min_eigenfrequency_THz" in audit_df.columns:
    values = pd.to_numeric(audit_df["eigen_true_min_eigenfrequency_THz"], errors="coerce").dropna()
    if len(values):
        fig, ax = plt.subplots(figsize=(7.0, 5.0))
        ax.hist(values, bins=min(40, max(10, int(np.sqrt(len(values))))))
        for threshold in IMAGINARY_THRESHOLDS_THz:
            ax.axvline(threshold, linestyle="--", linewidth=1)
        ax.set_xlabel("Minimum unbroadened mesh eigenfrequency (THz)")
        ax.set_ylabel("Number of structures")
        ax.set_title("True harmonic-instability diagnostic")
        ax.grid(axis="y", alpha=0.25)
        save_figure(fig, "05_true_minimum_eigenfrequency_distribution")

# ----------------------------- MACHINE-READABLE CONFIG -----------------------
config = {
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "train_pdos_root": str(TRAIN_PDOS_ROOT),
    "test_pdos_root": str(TEST_PDOS_ROOT),
    "search_roots": [str(x) for x in search_roots],
    "n_option_A_spectra": len(selected_spectra),
    "thresholds": {
        "pass_max_relative_stoich_error": PASS_MAX_RELATIVE_STOICH_ERROR,
        "warn_max_relative_stoich_error": WARN_MAX_RELATIVE_STOICH_ERROR,
        "pass_max_absolute_count_error_atoms": PASS_MAX_ABSOLUTE_COUNT_ERROR_ATOMS,
        "warn_max_absolute_count_error_atoms": WARN_MAX_ABSOLUTE_COUNT_ERROR_ATOMS,
        "pass_total_variation_error": PASS_TOTAL_VARIATION_ERROR,
        "warn_total_variation_error": WARN_TOTAL_VARIATION_ERROR,
        "Li_pass_max_absolute_count_error_atoms": LI_PASS_MAX_ABSOLUTE_COUNT_ERROR_ATOMS,
        "Li_warn_max_absolute_count_error_atoms": LI_WARN_MAX_ABSOLUTE_COUNT_ERROR_ATOMS,
        "Li_stoichiometry_fail_threshold": LI_STOICHIOMETRY_FAIL_THRESHOLD,
        "anchor_min_expected_count": ANCHOR_MIN_EXPECTED_COUNT,
        "anchor_min_fraction_of_max": ANCHOR_MIN_FRACTION_OF_MAX,
        "severe_min_distance_A": SEVERE_MIN_DISTANCE_A,
        "warning_min_distance_A": WARNING_MIN_DISTANCE_A,
        "coarse_DOS_grid_THz": COARSE_DOS_GRID_THz,
        "high_max_frequency_THz": HIGH_MAX_FREQUENCY_THz,
        "very_high_max_frequency_THz": VERY_HIGH_MAX_FREQUENCY_THz,
        "imaginary_thresholds_THz": IMAGINARY_THRESHOLDS_THz,
        "minimum_automatic_production_score": MIN_AUTOMATIC_PRODUCTION_SCORE,
        "minimum_automatic_source_score": MIN_AUTOMATIC_SOURCE_SCORE,
    },
    "manual_overrides": STRUCTURE_OVERRIDES,
}
(LOGS / "audit_configuration.json").write_text(json.dumps(config, indent=2), encoding="utf-8")

# Manual override template for ambiguous selections.
override_template = pd.DataFrame([
    {
        "ID": "replace_id",
        "production_structure_file": "/content/drive/MyDrive/path/ultra_relaxed_generated.cif",
        "source_structure_file": "/content/drive/MyDrive/path/ordered_adaptive_ID.cif",
        "eigen_file": "/content/drive/MyDrive/path/phonon_eigen_data.npz",
        "spectrum_file": "/content/drive/MyDrive/path/Option_A.csv",
        "notes": "Copy paths into STRUCTURE_OVERRIDES and rerun",
    }
])
override_template.to_csv(TABLES / "manual_override_template.csv", index=False)

# ------------------------------ TEXT SUMMARY --------------------------------
summary_lines = [
    "# OBELiX production-structure composition audit",
    "",
    f"Generated: {config['generated_utc']}",
    f"Option-A spectra audited: {len(audit_df)}",
    "",
    "## Audit status counts",
]
if not audit_df.empty:
    for status, count in audit_df["audit_status"].value_counts(dropna=False).items():
        summary_lines.append(f"- {status}: {count}")
summary_lines.extend([
    "",
    "## Interpretation rules",
    "- PASS means exact scale-proportional agreement with the official OBELiX composition.",
    "- PASS_INTEGERIZED_COMPOSITION means the finite ordered cell differs only by nearest-integer atom-count rounding or <=2% relative residuals, with <=2% total-variation composition error.",
    "- WARN_INTEGERIZATION_APPROXIMATION means the ordered finite cell requires up to a one-atom adjustment for at least one species but remains within the configured 6% total-variation limit.",
    "- FAIL_LI_STOICHIOMETRY means the non-Li framework is acceptable but Li is not, consistent with vacancy over/under-filling.",
    "- source_to_production_ordering_audit.csv compares ordered_adaptive_<ID>.cif with relaxed_<ID>.cif and therefore tests whether relaxation changed atom counts.",
    "- Negative plotted-DOS grid endpoints are not used as stability evidence.",
    "- True instability diagnostics come from phonon_eigen_data.npz mesh frequencies.",
    "- The oxidation-state result is only a feasibility heuristic, not a chemical validation.",
    "",
    "## Primary files",
    "- tables/production_structure_composition_audit.csv",
    "- tables/source_to_production_ordering_audit.csv",
    "- tables/spectrum_grid_qc.csv",
    "- tables/eigenfrequency_stability_qc.csv",
    "- tables/regeneration_priority.csv",
])
(LOGS / "AUDIT_SUMMARY.md").write_text("\n".join(summary_lines), encoding="utf-8")

# ------------------------------- COPY TO DRIVE -------------------------------
OUTPUT_DRIVE_DIR.mkdir(parents=True, exist_ok=True)
for child in OUTPUT_DRIVE_DIR.iterdir():
    if child.is_dir():
        shutil.rmtree(child)
    else:
        child.unlink()
for item in LOCAL_WORKDIR.iterdir():
    destination = OUTPUT_DRIVE_DIR / item.name
    if item.is_dir():
        shutil.copytree(item, destination)
    else:
        shutil.copy2(item, destination)

zip_local = Path("/content") / ZIP_NAME
if zip_local.exists():
    zip_local.unlink()
with zipfile.ZipFile(zip_local, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in LOCAL_WORKDIR.rglob("*"):
        if path.is_file():
            archive.write(path, arcname=path.relative_to(LOCAL_WORKDIR))
zip_drive = OUTPUT_DRIVE_DIR.parent / ZIP_NAME
shutil.copy2(zip_local, zip_drive)

print("\n" + "=" * 88)
print("AUDIT COMPLETE")
print("=" * 88)
print(summary_df.to_string(index=False))
print(f"\nResults folder: {OUTPUT_DRIVE_DIR}")
print(f"ZIP archive:    {zip_drive}")
print("\nInspect regeneration_priority.csv first, then verify automatic file selections")
print("in all_structure_candidates.csv before making irreversible dataset decisions.")


Mounted at /content/drive
TRAIN: /content/drive/MyDrive/MLIP-main/DOSFILES/Train_cif
  Li PDOS files: 249
  Total DOS files: 249
TEST: /content/drive/MyDrive/MLIP-main/DOSFILES/Test_cif
  Li PDOS files: 64
  Total DOS files: 64
Option-A spectra found: 311
Indexing auxiliary files under: /content/drive/MyDrive/MLIP-main/DOSFILES/Train_cif
Indexing auxiliary files under: /content/drive/MyDrive/MLIP-main/DOSFILES/Test_cif
Indexing auxiliary files under: /content/drive/MyDrive/Train_cif
[  1/311] test/0ii
[  2/311] test/1ty
[  3/311] test/24k
[  4/311] test/2d1
[  5/311] test/2r7
[  6/311] test/2uv
[  7/311] test/3rh
[  8/311] test/4ie
[  9/311] test/4p7
[ 10/311] test/7cr
[ 11/311] test/9n0
[ 12/311] test/9o6
[ 13/311] test/aab
[ 14/311] test/ar1
[ 15/311] test/ard
[ 16/311] test/b56
[ 17/311] test/bnu
[ 18/311] test/br1
[ 19/311] test/cfy
[ 20/311] test/cti
[ 21/311] test/cz2
[ 22/311] test/de5
[ 23/311] test/duf
[ 24/311] test/dvb
[ 25/311] test/ecp
[ 26/311] test/ei3
[ 27/311] test/elz